In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

DATASET_ROOT = Path("/root/Projet_Image/SH17dataset")
RUNS_DIR     = Path("/root/Projet_Image/runs")
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi' / 'weights' / 'best.pt'
DEVICE       = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(BEST_WEIGHTS))
print(f"Modèle chargé : {BEST_WEIGHTS}")
print(f"Device        : {DEVICE}")

In [ ]:
results = model.val(
    data    = str(DATASET_ROOT / 'sh17.yaml'),
    split   = 'test',
    imgsz   = 640,
    batch   = 16,
    device  = DEVICE,
    verbose = False,
    plots   = False,
)

In [ ]:
map50     = float(results.box.map50)
map50_95  = float(results.box.map)
precision = float(results.box.mp)
recall    = float(results.box.mr)
f1_global = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"mAP50      : {map50:.3f}")
print(f"mAP50-95   : {map50_95:.3f}")
print(f"Précision  : {precision:.3f}")
print(f"Rappel     : {recall:.3f}")
print(f"F1 (global): {f1_global:.3f}")

In [ ]:
names       = model.names
class_idx   = results.box.ap_class_index
precision_c = results.box.p
recall_c    = results.box.r
f1_c        = results.box.f1
ap50_c      = results.box.ap50
ap_c        = results.box.ap

rows = [
    {
        'Classe':    names[i],
        'Précision': round(float(p), 3),
        'Rappel':    round(float(r), 3),
        'F1':        round(float(f), 3),
        'mAP50':     round(float(a50), 3),
        'mAP50-95':  round(float(a), 3),
    }
    for i, p, r, f, a50, a in zip(class_idx, precision_c, recall_c, f1_c, ap50_c, ap_c)
]
rows.append({
    'Classe':    'GLOBAL (moyenne)',
    'Précision': round(precision, 3),
    'Rappel':    round(recall, 3),
    'F1':        round(f1_global, 3),
    'mAP50':     round(map50, 3),
    'mAP50-95':  round(map50_95, 3),
})

table = pd.DataFrame(rows)
table